In [ ]:
import pandas as pd
import sys

sys.path.append("prior_code")

## Load the CSV files

In [ ]:
# Load all three CSV files
python_cols = pd.read_csv("prior_code/siab_python_colnames.csv")
stata_cols = pd.read_csv("prior_code/siab_afterStata_colnames.csv")
processed_cols = pd.read_csv("prior_code/siab_processed_colnames.csv")

print(f"Python columns: {len(python_cols)}")
print(f"Stata columns: {len(stata_cols)}")
print(f"Processed columns: {len(processed_cols)}")

## Colname Mapping


In [ ]:
# Mapping from Python column names to Stata column names
# siab_python_colnames.csv => siab_afterStata_colnames.csv
PYTHON_TO_STATA = {
    "person_id": "persnr",
    "year": "year",
    "spell_source": "quelle_gr",
    "episode_start_date": "begepi",
    "episode_end_date": "endepi",
    "unemployment_duration_days": "alo_dau",
    "ue_pop": "ue_pop",
    "nrEntry": "nrEntry",
    "beg_ue": "beg_ue",
    "ue_duration": "ue_duration",
    "ltue": "ltue",
    "employed_before": "employed_before",
    "receipt_leh_before": "receipt_leh_before",
    "receipt_lhg_before": "receipt_lhg_before",
    "se_before": "se_before",
    "ASU_notue_seeking_before": "ASU_notue_seeking_before",
    "ASU_other_before": "ASU_other_before",
    "break_before": "break_before",
    "tsince_ein_erw1": "tsince_ein_erw1",
    "tsince_ein_erw1_cat": "tsince_ein_erw1_cat",
    "tsince_ft_lm_contact": "tsince_ft_lm_contact",
    "tsince_lm_contact": "tsince_lm_contact",
    "tsince_lm_contact_cat": "tsince_lm_contact_cat",
    "tsince_ft_lm_contact_cat": "tsince_ft_lm_contact_cat",
    "emp_total": "emp_total",
    "est_total": "est_total",
    "emp1_total": "emp1_total",
    "emp1_total_dur": "emp1_total_dur",
    "emp2_total_dur": "emp2_total_dur",
    "emp3_total_dur": "emp3_total_dur",
    "emp1_m_dur": "emp1_m_dur",
    "lastjob_none": "lastjob_none",
    "lastjob_tot_dur": "lastjob_tot_dur",
    "lastjob_type": "lastjob_type",
    "lastjob_pt": "lastjob_pt",
    "lastjob_niveau": "lastjob_niveau",
    "lastjob_leih": "lastjob_leih",
    "lastjob_befrist": "lastjob_befrist",
    "lastjob_educ": "lastjob_educ",
    "lastjob_ao_kreis": "lastjob_ao_kreis",
    "age": "age",
    "female": "frau",
    "education_level_imputed": "ausbildung_imp",
    "maxgerman_citizen": "maxdeutsch",
    "maxeducation_level_imputed": "maxausbildung_imp",
    "maxschool_completed": "maxschule",
    "maxoccupation_skill_level": "maxniveau",
    "maxbula": "maxbula",
    "maxcommuter": "maxpendler",
    "lastwage_group": "lastw08_gen_gr",
    "occupation_skill_level": "niveau",
}

# Reverse mapping from Stata column names to Python column names
STATA_TO_PYTHON = {v: k for k, v in PYTHON_TO_STATA.items()}

## Extract column names as sets

In [ ]:
# Get sets of column names
python_set = set(python_cols["colname"].dropna())
stata_set = set(stata_cols["colname"])
processed_set = set(processed_cols["colname"])

print(f"Unique Python columns: {len(python_set)}")
print(f"Unique Stata columns: {len(stata_set)}")
print(f"Unique Processed columns: {len(processed_set)}")

## Apply Mapping to Python Columns

Convert Python column names to their Stata equivalents using the mapping.

In [ ]:
# Map Python columns to Stata names
python_cols["mapped_to_stata"] = python_cols["colname"].map(PYTHON_TO_STATA)

# Identify which Python columns are in the mapping
mapped_cols = python_cols[python_cols["mapped_to_stata"].notna()]
unmapped_cols = python_cols[python_cols["mapped_to_stata"].isna()]

print(f"Python columns with mapping: {len(mapped_cols)}")
print(f"Python columns without mapping: {len(unmapped_cols)}")

if len(unmapped_cols) > 0:
    print("\nUnmapped Python columns:")
    print(unmapped_cols["colname"].tolist())

## Compare Python (mapped) with Stata Columns

In [ ]:
# Get mapped Python column names
mapped_python_set = set(mapped_cols["mapped_to_stata"])

# Columns in Stata but not in mapped Python
stata_only = stata_set - mapped_python_set
print(f"Columns only in Stata (not in mapped Python): {len(stata_only)}")
if stata_only:
    print(sorted(stata_only))

print("\n" + "=" * 80 + "\n")

# Columns in mapped Python but not in Stata
python_only = mapped_python_set - stata_set
print(f"Columns in mapped Python but not in Stata: {len(python_only)}")
if python_only:
    print(sorted(python_only))

print("\n" + "=" * 80 + "\n")

# Common columns
common_python_stata = mapped_python_set & stata_set
print(f"Common columns between mapped Python and Stata: {len(common_python_stata)}")

## Compare Python (mapped) with Processed Columns

In [ ]:
# Columns in Python (mapped) but not in Processed
python_not_processed = mapped_python_set - processed_set
print(
    f"Columns only in Python (mapped) (not in Processed): {len(python_not_processed)}"
)
if python_not_processed:
    print(sorted(python_not_processed))

print("\n" + "=" * 80 + "\n")

# Columns in Processed but not in Python (mapped)
processed_not_python = processed_set - mapped_python_set
print(f"Columns in Processed but not in Python (mapped): {len(processed_not_python)}")
if processed_not_python:
    print(sorted(processed_not_python))

print("\n" + "=" * 80 + "\n")

# Common columns
common_python_processed = mapped_python_set & processed_set
print(
    f"Common columns between Python (mapped) and Processed: {len(common_python_processed)}"
)

## Create Summary DataFrames

Create detailed comparison tables showing the relationships between column names.

In [ ]:
# Create a comprehensive comparison DataFrame
comparison_data = []

# Start with Python columns and their mappings
for _, row in python_cols.iterrows():
    python_name = row["colname"]
    stata_name = row["mapped_to_stata"]

    comparison_data.append(
        {
            "name_python": python_name,
            "name_stata": stata_name if pd.notna(stata_name) else None,
            "in_ours": True,
            "in_postStata": stata_name in stata_set if pd.notna(stata_name) else False,
            "in_processedR": stata_name in processed_set
            if pd.notna(stata_name)
            else False,
        }
    )

# Add Stata columns that don't have Python equivalents
for stata_name in stata_only:
    comparison_data.append(
        {
            "name_python": None,
            "name_stata": stata_name,
            "in_ours": False,
            "in_postStata": True,
            "in_processedR": stata_name in processed_set,
        }
    )

# Add columns that are only in processedR (not in Python or Stata)
processed_only = processed_set - mapped_python_set - stata_set
for processed_name in processed_only:
    comparison_data.append(
        {
            "name_python": None,
            "name_stata": processed_name,
            "in_ours": False,
            "in_postStata": False,
            "in_processedR": True,
        }
    )

comparison_df = pd.DataFrame(comparison_data)

# Sort by presence in datasets
comparison_df.sort_values(
    by=["in_ours", "in_postStata", "in_processedR", "name_python", "name_stata"],
    ascending=False,
    inplace=True,
)
comparison_df.to_csv("prior_code/colname_comparison.csv", index=False)

# Filter to show only columns missing in Python version
missing_in_python = comparison_df[comparison_df["in_ours"] == False].copy()
print(f"Columns missing in Python version: {len(missing_in_python)}")
print(f"  - Present in postStata: {missing_in_python['in_postStata'].sum()}")
print(f"  - Present in processedR: {missing_in_python['in_processedR'].sum()}")
print(
    f"  - Only in processedR: {(~missing_in_python['in_postStata'] & missing_in_python['in_processedR']).sum()}"
)
missing_in_python

## Summary Statistics

In [ ]:
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(
    f"\nTotal unique columns across all sources: {len(python_set | stata_set | processed_set)}"
)
print(
    f"\nColumns in all three sources: {len(mapped_python_set & stata_set & processed_set)}"
)
print(
    f"Columns in Python and Stata only: {len((mapped_python_set & stata_set) - processed_set)}"
)
print(
    f"Columns in Stata and Processed only: {len((stata_set & processed_set) - mapped_python_set)}"
)
print(f"Columns unique to Python: {len(mapped_python_set - stata_set - processed_set)}")
print(f"Columns unique to Stata: {len(stata_set - mapped_python_set - processed_set)}")
print(
    f"Columns unique to Processed: {len(processed_set - stata_set - mapped_python_set)}"
)

## Check for Naming Issues

Identify potential issues in the mapping or column names.

In [ ]:
# Check if any mapped columns don't exist in Stata
mapped_not_in_stata = [
    col for col in mapped_cols["mapped_to_stata"] if col not in stata_set
]
print(f"Mapped columns that don't exist in Stata file: {len(mapped_not_in_stata)}")
if mapped_not_in_stata:
    for col in mapped_not_in_stata:
        python_name = mapped_cols[mapped_cols["mapped_to_stata"] == col][
            "colname"
        ].iloc[0]
        print(f"  {python_name} -> {col}")